In [1]:
# Cellule 1 : Imports et configuration
%load_ext autoreload
%autoreload 2

import tools.ai_token as tk
import tools.ai_mlx as ai
import tools.ai_mlx_claude as aiC

import mlx.core as mx
import mlx.nn as nn
import numpy as np
from mlx.optimizers import AdamW
import time


In [ ]:
%reload_ext autoreload

In [2]:
fileName = 'data/corpus_Moliere.txt'
addedToken = 100  # Nombre de tokens désirés (bytes 0-255)
token = tk.BPETokenizer(fileName, addToken=addedToken)
token.train_optimiser_v2(False)
text_token = token.ids

Nettoyage terminé:


In [ ]:
print(mx.default_device()) # Devrait afficher Device(gpu, 0)

In [ ]:
# --- INITIALISATION ET TEST ---
config = ai.Config()
config.vocab_size = len(token.vocab)
dataset = ai.MakeDataset(token.ids)
model = ai.Decoder(config)
mx.eval(model.parameters()) # Force l'initialisation sur le GPU
model.set_dtype(mx.float16)

# Setup optimizer
optimizer = AdamW(learning_rate=config.learning_rate)

# Fonction d'entraînement compilée pour le M1
#loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

# 1. Conversion en array MLX (équivalent de torch.tensor)
# On utilise int32 car c'est le standard pour les indices en MLX
data = mx.array(token.ids, dtype=mx.int32)

# 2. Split Train/Val (Le slicing en MLX ne copie pas les données, c'est une "vue")
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

trainer = ai.TransformerTrainer(model=model, optimizer=optimizer,dataset_obj=dataset,config=config)
trainer.run(config.max_iters, config.eval_iterval)


In [ ]:
def init_weights(model):
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear):
            # Initialisation de Xavier / Glorot
            m.weight = mx.random.normal(m.weight.shape) * 0.02
        elif isinstance(m, nn.Embedding):
            m.weight = mx.random.normal(m.weight.shape) * 0.02

In [ ]:
# --- INITIALISATION ET TEST ---
config = ai.Config()
# text_token = token.encode(token.text_cleaned)
config.vocab_size = len(token.vocab)
# On booste le learning rate car le vocab est petit
config.learning_rate = 1e-4

dataset = ai.MakeDataset(text_token)
model = ai.Decoder(config)
init_weights(model)

# 1. On RESTE en float32 pour l'entraînement (plus stable)
model.set_dtype(mx.float16) # <--- Enlève ou commente cette ligne

# 2. Initialisation forcée
mx.eval(model.parameters())

# 3. Optimizer
optimizer = AdamW(learning_rate=config.learning_rate, weight_decay=0.01)

# 4. Lancement
trainer = ai.TransformerTrainer(model=model, optimizer=optimizer, dataset_obj=dataset, config=config)
trainer.run(config.max_iters,config.eval_iterval)


In [ ]:
# Exemple d'usage dans ton Jupyter :
context = mx.array([token.encode("Il était une fois")])
print(context)
out = model.generate(context, max_new_tokens=50, temperature=1.)
print(token.decode(out))


In [ ]:
# Force un batch unique pour tester la capacité d'apprentissage
x_test, y_test = trainer.get_batch("train") 

for i in range(500):
    loss = trainer.compiled_eval_step(x_test, y_test)
    mx.eval(loss)
    if i % 50 == 0:
        print(f"Test Overfit - Step {i}: Loss {loss.item():.4f}")

In [ ]:
# À tester lors de l'initialisation de tes données
x = x.astype(mx.float16)

In [ ]:
# --- ASTUCE PERFORMANCE M1 ---
# Définir une fonction de perte ou de forward complète et la compiler
# model = CausalSelfAttention(config)
# model = mx.compile(model) # <--- C'est ici que la magie opère

In [5]:
# --- INITIALISATION ET TEST ---
config = aiC.Config()
# text_token = token.encode(token.text_cleaned)
config.vocab_size = len(token.vocab)
# On booste le learning rate car le vocab est petit
config.learning_rate = 1e-4

dataset = aiC.MakeDataset(text_token)
model = aiC.Decoder(config)
# init_weights(model)

# 1. On RESTE en float32 pour l'entraînement (plus stable)
model.set_dtype(mx.float16) # <--- Enlève ou commente cette ligne

# 2. Initialisation forcée
mx.eval(model.parameters())

# 3. Optimizer
optimizer = AdamW(learning_rate=config.learning_rate, weight_decay=0.01)

# 4. Lancement
trainer = ai.TransformerTrainer(model=model, optimizer=optimizer, dataset_obj=dataset, config=config)
trainer.run(config.max_iters,config.eval_interval)


Dataset : Train 863707 | Val 95968


AttributeError: module 'tools.ai_mlx_claude' has no attribute 'Decoder'